In [1]:
!pip install datasets

   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 3.4 MB/s  0:00:00
   ---------------------------------------- 0.0/784.9 kB ? eta -:--:--
   ---------------------------------------- 784.9/784.9 kB 3.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------------ --------------------------- 1.3/4.0 MB 6.9 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.0 MB 5.9 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 5.3 MB/s  0:00:00

  Attempting uninstall: dill

    Found existing installation: dill 0.4.0

    Uninstalling dill-0.4.0:

      Successfully uninstalled dill-0.4.0

   ----------- ---------------------------- 2/7 [dill]
   ----------- ---------------------------- 2/7 [dill]
   ----------- ---------------------------- 2/7 [dill]
   ----------- -----

In [3]:
from datasets import load_dataset

# Load WikiText-2 directly from the official repo
dataset = load_dataset("salesforce/wikitext", "wikitext-2-v1")

# Print structure
print(dataset)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

D:\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Miruthula M\.cache\huggingface\hub\datasets--salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


wikitext-2-v1/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  685kB            

wikitext-2-v1/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

wikitext-2-v1/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 6.07MB            

...

wikitext-2-v1/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [6]:
import re
from collections import defaultdict, Counter

# Step 1: Extract Text from Hugging Face Dataset (loaded in Cell [3])
raw_text = " ".join(dataset['train']['text'])
words = re.findall(r"[a-zA-Z]+", raw_text.lower())

print("WikiText-2 Corpus Loaded Successfully!")
print(f"Total Words Extracted: {len(words)}")

# Step 2: Build Bigram Model (or Bigram/Trigram lookup)
bigram_counts = defaultdict(Counter)

# Count word transitions (w1 -> w2)
for w1, w2 in zip(words[:-1], words[1:]):
    bigram_counts[w1][w2] += 1

# Step 3: Define predict_next_words Function
def predict_next_words(test_words, top_n=5):
    if not test_words:
        return []
    
    # Get the last word from the input sentence
    last_word = test_words[-1]
    
    # Look up predictions for the last word
    next_word_counts = bigram_counts.get(last_word, {})
    total_count = sum(next_word_counts.values())
    
    if total_count == 0:
        return []
    
    # Calculate probabilities and sort top N
    predictions = [
        (word, count / total_count)
        for word, count in next_word_counts.most_common(top_n)
    ]
    return predictions

# Step 4: Test with Different Sentences
print("\n======================================")
print("Testing Different Sentences")
print("======================================")

test_sentences = [
    "machine learning",
    "artificial intelligence",
    "the united",
    "new york",
    "in the"
]

for test_sentence in test_sentences:
    test_words = re.findall(r"[a-zA-Z]+", test_sentence.lower())
    predictions = predict_next_words(test_words, top_n=5)
    
    print("\nInput:", test_sentence)
    print("Predictions:")
    if not predictions:
        print("  No predictions found.")
    else:
        for i, (word, probability) in enumerate(predictions, start=1):
            print(f"  {i} . {word} - {round(probability, 3)}")

print("\nNext-Word Prediction System Completed Successfully!")

WikiText-2 Corpus Loaded Successfully!
Total Words Extracted: 1694562

Testing Different Sentences

Input: machine learning
Predictions:
  1 . that - 0.133
  2 . curve - 0.093
  3 . the - 0.08
  4 . to - 0.08
  5 . about - 0.067

Input: artificial intelligence
Predictions:
  1 . and - 0.046
  2 . management - 0.046
  3 . on - 0.037
  4 . unk - 0.037
  5 . ai - 0.037

Input: the united
Predictions:
  1 . states - 0.644
  2 . kingdom - 0.138
  3 . nations - 0.046
  4 . in - 0.024
  5 . on - 0.012

Input: new york
Predictions:
  1 . city - 0.128
  2 . times - 0.12
  3 . in - 0.047
  4 . s - 0.045
  5 . state - 0.044

Input: in the
Predictions:
  1 . unk - 0.035
  2 . first - 0.017
  3 . th - 0.01
  4 . song - 0.009
  5 . game - 0.007

Next-Word Prediction System Completed Successfully!
